## **1. Projeto de Inteligência Artificial 2025/2026: Agentes para o Jogo PopOut**

**Faculdade de Ciências da Universidade do Porto (FCUP)** **Unidade Curricular:** Inteligência Artificial (CC2006)  
**Turma:** CC2006_PL1 | **Grupo:** 3  
**Docente:** Prof. Francesco Renna  

### **Equipe de Desenvolvimento**
| Número | Nome |
| :--- | :--- |
| **202300654** | Augusto Moreira |
| **202300276** | Guilherme Klippel |
| **202300916** | Yan Coelho |

---

### **Introdução e Contexto do Problema**

Este notebook documenta a conceção, implementação e análise de agentes de Inteligência Artificial para o jogo **PopOut**, uma variante estratégica e dinâmica do clássico *Connect-4* (Quatro em Linha). Tratando-se de um jogo de tabuleiro de soma nula e informação perfeita, o PopOut apresenta um desafio acrescido devido à sua mecânica de remoção de peças, que aumenta significativamente a complexidade e o fator de ramificação (*branching factor*) da árvore de jogo.

O objetivo deste projeto divide-se em duas abordagens fundamentais da IA:
1. **Procura Adversarial:** Implementação de um agente baseado em **Monte Carlo Tree Search (MCTS)** para atuar como um jogador de elite (*expert*).
2. **Aprendizagem Automática:** Utilização do MCTS para gerar conjuntos de dados de alta qualidade, servindo de base para o treino de um modelo de **Árvores de Decisão (algoritmo ID3)**, capaz de inferir regras táticas a partir das jogadas guardadas nos datasets.

### **Mecânicas e Regras Especiais**

A nossa implementação na classe base `PopOutGame` suporta o tabuleiro normal de 6x7 e introduz as seguintes ações:
* **Drop:** Colocação de uma peça no topo de uma coluna, caindo até à posição livre mais baixa.
* **PopOut:** Remoção de uma peça do próprio jogador na base (linha inferior) de uma coluna, fazendo com que todas as peças acima desçam uma posição.

Para garantir a robustez das simulações, foram rigorosamente implementadas as **três regras especiais** exigidas no guião:
* **Regra 1 (Vitória Simultânea):** Se uma jogada *PopOut* alinhar simultaneamente 4 peças para ambos os jogadores, a vitória é atribuída ao jogador que efetuou o movimento.
* **Regra 2 (Tabuleiro Cheio):** Caso a linha superior do tabuleiro esteja totalmente preenchida, o movimento *Drop* torna-se inválido, forçando o jogador a executar um *PopOut* (ou declarar derrota caso não tenha peças na base).
* **Regra 3 (Empate por Repetição):** O sistema monitoriza o histórico de estados (através de *hashes* do tabuleiro). Se o mesmo estado se repetir 3 vezes durante a partida, o jogo é declarado como empate.

---

#### Importes

In [ ]:
%load_ext autoreload
%autoreload 2

# Bibliotecas padrão
import os
import warnings
import pickle
import itertools
import time

# Bibliotecas externas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos locais
from src.game import (
    PopOutGame,
    _get_ai_move,
    PLAYER1,
    PLAYER2,
    main_menu
)

from src.decision_tree import (
    DecisionTreeID3,
    clean_conflicting_data,
    discretizar_largura_igual,
    discretizar_frequencia_igual,
    calcular_metricas,
    plotar_arvore_decisao
)

from src.mcts import MCTS
from src.dataset_generator import run_batch_simulation

# Configurações globais
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
os.makedirs("datasets", exist_ok=True)

## **2. Geração de Datasets (MCTS)**
Nesta etapa, utilizamos simulações de **Monte Carlo Tree Search (MCTS)** (IA vs IA) para gerar um histórico de jogadas de alta qualidade. Estes dados servirão de base para o treino posterior do classificador ID3.

* **Simulação Automática (`run_batch_simulation`)**: Coordena as partidas garantindo a aplicação estrita das regras (PopOut, Vitória Dupla, Empate por Repetição).
* **Rastreabilidade**: O nome dos ficheiros CSV é gerado dinamicamente com base nos hiperparâmetros usados: `it` (Iterações), `c` (Constante de Exploração) e `mc` (Max Children).
* **Estrutura dos Dados**: Cada registo no dataset captura um estado do jogo, contendo as 42 posições do tabuleiro (*features*) e a jogada ótima decidida (*target*).

In [ ]:
# Configuração das instâncias MCTS para treino
ia_mcts1 = MCTS(iterations=100, c=1.41, max_children=None, max_depth=None, pure_mode=True)
ia_mcts2 = MCTS(iterations=100, c=1.41, max_children=None, max_depth=None, pure_mode=True)



# Execução dos lotes de simulação
run_batch_simulation(num_games=200, ia_1=ia_mcts1, ia_2=ia_mcts2)

print("Processo concluído. Ficheiros guardados na pasta 'datasets/'.")

## **3. Treino, Visualização e Geração da Árvore de Decisão**
Nesta etapa, utilizamos o algoritmo **ID3** para transformar os dados brutos gerados pelo MCTS num classificador de decisões lógicas. 

O grande destaque desta fase é a implementação da **Normalização de Perspetiva Relativa**. Em vez de a Árvore aprender táticas separadas para o "Jogador 1" e "Jogador 2", o dataset é convertido para uma matriz universal de "Eu (1)" vs "Inimigo (-1)". Esta abordagem:
* Reduz drasticamente o espaço de estados que a árvore precisa de memorizar.
* Maximiza a eficácia do cálculo de Entropia e Ganho de Informação.
* Permite ao modelo generalizar táticas avançadas independentemente de quem tem a iniciativa de jogo.

**Gerador da Àrvore**

In [ ]:
# =====================================================================
# FUNÇÃO: TREINAR E GUARDAR A ÁRVORE (Direto, sem avaliação)
# =====================================================================
def treinar_e_guardar_besta(caminho_dataset, nome_arvore_final, only_winners=True, max_depth=100):
    print(f"\n--- A PREPARAR DADOS PARA O MODELO: {nome_arvore_final} ---")
    df = pd.read_csv(caminho_dataset)
    
    # Filtrar apenas vitórias, se solicitado
    if only_winners:
        df = df[(df['winner'] != 0) & (df['p_turn'] == df['winner'])]
        
    # Normalizar o tabuleiro (1 para a própria peça, -1 para o adversário, 0 vazio)
    colunas_tabuleiro = [f"pos_{i}" for i in range(42)]
    for col in colunas_tabuleiro:
        df[col] = np.where(df[col] == 0, 0, 
                  np.where(df[col] == df['p_turn'], 1, -1))
        
    # Criar a variável alvo (ex: "3_d", "2_p") e limpar colunas
    df["target_move"] = df["col"].astype(str) + "_" + df["type"]
    df_clean = df.drop(columns=["game_id", "winner", "col", "type", "p_turn"])
    df_clean = clean_conflicting_data(df_clean, target_name="target_move")
    
    # TREINO DIRETO COM 100% DOS DADOS
    print(f"\n[A TREINAR] A forjar a versão final da IA com 100% dos dados ({len(df_clean)} linhas)...")
    arvore_100 = DecisionTreeID3(max_depth=max_depth)
    arvore_100.fit(df_clean, target_name="target_move")
    
    # GUARDAR O MODELO
    os.makedirs("modelos_treinados", exist_ok=True)
    caminho_ficheiro = f"modelos_treinados/{nome_arvore_final}.pkl"
    with open(caminho_ficheiro, 'wb') as f:
        pickle.dump(arvore_100, f)
        
    print(f"[SUCESSO] Árvore guardada para a Arena em: {caminho_ficheiro}")

# ------ EXECUÇÃO DO TREINO ------
DATASET = "datasets/P1_it100_c1.41_mcAll_dNone_pure_vs_P2_it100_c1.41_mcAll_dNone_pure.csv" 

treinar_e_guardar_besta(
    caminho_dataset=DATASET,
    nome_arvore_final="worst_tree_all",
    only_winners=False,
    max_depth=100
)

**Só visualização da Àrvore completa**


In [ ]:

# =====================================================================
# FUNÇÕES DE APOIO E VISUALIZAÇÃO
# =====================================================================
def calcular_profundidade(node):
    """Calcula a profundidade máxima da árvore de forma recursiva."""
    if not node.children:
        return 0
    return 1 + max([calcular_profundidade(filho) for filho in node.children.values()])

def corrigir_arvore_antiga(node):
    """Garante que o nó possui o atributo branch_counts para evitar erros de plotagem."""
    if not hasattr(node, 'branch_counts'):
        node.branch_counts = {}
    for filho in node.children.values():
        corrigir_arvore_antiga(filho)

def visualizar_arvore_existente(nome_arvore):
    """
    Carrega um ficheiro .pkl e gera o seu diagrama visual.
    Foca-se na estrutura e profundidade dos nós.
    """
    caminho_pkl = f"modelos_treinados/{nome_arvore}.pkl"
    
    try:
        with open(caminho_pkl, 'rb') as f:
            arvore = pickle.load(f)
            
        corrigir_arvore_antiga(arvore.root)
        prof = calcular_profundidade(arvore.root)
        print(f"[OK] Árvore '{nome_arvore}' carregada. (Profundidade: {prof})")
        
        metricas_visuais = {
            'profundidade': prof,
            'total': 'N/A (Modo Visual)',
            'corretas': 'N/A',
            'acuracia': 0.0
        }
        
        plotar_arvore_decisao(arvore.root, 
                              titulo=f"Estrutura da Árvore: {nome_arvore}", 
                              metricas=metricas_visuais)
                              
    except FileNotFoundError:
        print(f"[ERRO] Não encontrei o ficheiro '{caminho_pkl}'.")

visualizar_arvore_existente("worst_tree_all")

## **4. Fase de Análise Exploratória de Dados**

Esta fase é dedicada à auditoria visual do dataset gerado pelo **MCTS**. Antes de alimentar a Árvore de Decisão (**ID3**), precisamos de garantir que a informação estratégica capturada faz sentido. Com a introdução do rastreamento de vitórias (`winner` e `game_id`), a nossa análise deixa de observar apenas o volume geral e foca-se puramente na extração de táticas bem-sucedidas.

### Objetivos da Visualização
* **Qualidade dos Dados:** Garantir que o modelo ID3 irá aprender com instâncias de excelência estratégica, **removendo jogos empatados do processo de análise**.
* **Validação Tática:** Confirmar o peso estatístico do domínio da zona central do tabuleiro e o uso estratégico da mecânica de *PopOut*.
* **Contraste de Eficiência:** Isolar o comportamento das "Jogadas Vencedoras" contra as "Jogadas Perdedoras" para perceber o que separa uma IA comum de uma IA de elite.

---

### Dashboard de Performance e Estratégia
O script compila um painel com 8 painéis críticos focados no comportamento em partidas decisivas:

1. **Resultados Finais das Partidas**: Exibe a taxa de vitórias diretas (P1 vs P2), comprovando qual a configuração MCTS que obteve maior dominância.
2. **Volume Total de Jogadas**: Revela a quantidade de ações executadas no tabuleiro por cada lado.
3. **Distribuição de Colunas**: Identifica o volume global de jogadas em cada coluna.
4. **Proporção Global (Drop vs Pop)**: Analisa o rácio global de utilização da regra especial em relação à mecânica clássica.
5. **Foco Estratégico (Vencedores vs Perdedores)**: O indicador de maior valor para o algoritmo ID3. Isola e compara as colunas jogadas especificamente por quem, no fim, garantiu a vitória.
6. **Uso de Regras (Vencedores vs Perdedores)**: Demonstra se os vencedores recorrem mais vezes, ou de forma mais cirúrgica, à mecânica *PopOut* comparativamente aos derrotados.
7. **Mapa de Ocupação (Heatmap)**: Um mapa térmico da grelha (6x7) para mapear a concentração e disputa do território.
8. **Zonas de PopOut**: Detalha especificamente onde ocorrem as jogadas de remoção de peça, diferenciando o sucesso da ação.

In [ ]:
caminho_ficheiro = "datasets/P1_it100_c1.41_mcAll_dNone_pure_vs_P2_it100_c1.41_mcAll_dNone_pure.csv"
nome_do_ficheiro = os.path.basename(caminho_ficheiro)
df = pd.read_csv(caminho_ficheiro)

# --- A MUDANÇA ESTÁ AQUI: Não apagamos os empates e classificamos! ---
def get_status(row):
    if row['winner'] == 0:
        return 'Empate'
    elif row['p_turn'] == row['winner']:
        return 'Vencedor'
    else:
        return 'Perdedor'

df['status'] = df.apply(get_status, axis=1)

# Filtra apenas os jogos únicos
df_jogos = df.drop_duplicates(subset=['game_id'])[['game_id', 'winner']]

# CONTAGEM DO TOTAL DE PARTIDAS
total_partidas = len(df_jogos)

fig, axes = plt.subplots(4, 2, figsize=(18, 24))

# TÍTULO ATUALIZADO 
fig.suptitle(f"Análise Global e Estratégica: {nome_do_ficheiro}\nTotal de Partidas (Com Empates): {total_partidas}", fontsize=20, fontweight='bold', y=0.98)

# 1. Resultados Finais dos Jogos (Com Empate na posição 0)
sns.countplot(data=df_jogos, x='winner', hue='winner', order=[0, 1, 2], ax=axes[0, 0], palette=['#95a5a6', '#3498db', '#e74c3c'], legend=False)
axes[0, 0].set_title("1. Resultados Finais das Partidas")
axes[0, 0].set_xticklabels(['Empate', 'Vitória P1 (X)', 'Vitória P2 (O)'])

# 2. Total de Movimentos (Geral)
sns.countplot(data=df, x='p_turn', hue='p_turn', ax=axes[0, 1], palette=['#3498db', '#e74c3c'], legend=False)
axes[0, 1].set_title("2. Volume Total de Jogadas (P1 vs P2)")

# 3. Escolha de Colunas (Geral)
sns.countplot(data=df, x='col', hue='p_turn', ax=axes[1, 0], palette=['#3498db', '#e74c3c'])
axes[1, 0].set_title("3. Distribuição de Colunas: P1 vs P2")

# 4. Rácio Drop vs Pop (Geral)
tipos = df['type'].value_counts()
axes[1, 1].pie(tipos, labels=tipos.index, autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1, 1].set_title("4. Proporção Global: Drop (d) vs Pop (p)")

# --- Paleta de Cores consistente para os gráficos seguintes ---
status_palette = {'Vencedor': '#2ecc71', 'Perdedor': '#e67e22', 'Empate': '#95a5a6'}

# 5. Escolha de Colunas (Vencedor vs Perdedor vs Empate)
sns.countplot(data=df, x='col', hue='status', ax=axes[2, 0], palette=status_palette)
axes[2, 0].set_title("5. Foco Estratégico: Vencedores, Perdedores e Empates")

# 6. Uso da Mecânica (Vencedor vs Perdedor vs Empate)
sns.countplot(data=df, x='type', hue='status', ax=axes[2, 1], palette=status_palette)
axes[2, 1].set_title("6. Uso de Regras: Vencedores, Perdedores e Empates")

# 7. Heatmap de Ocupação (Geral)
colunas_tab = [f"pos_{i}" for i in range(42)]
ocupacao = (df[colunas_tab] > 0).mean().values.reshape(6, 7)
sns.heatmap(ocupacao, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[3, 0], cbar=False)
axes[3, 0].set_title("7. Mapa de Ocupação (Zonas Quentes)")

# 8. Onde ocorrem PopOuts? (Vencedor vs Perdedor vs Empate)
pops = df[df['type'] == 'p']
if not pops.empty:
    sns.countplot(data=pops, x='col', hue='status', ax=axes[3, 1], palette=status_palette)
    axes[3, 1].set_title("8. Zonas de PopOut: Sucesso, Falha e Impasse")
else:
    axes[3, 1].text(0.5, 0.5, "Sem jogadas PopOut", ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## **5. Teste no Iris com Discretização**
- **Teste A: Sem Discretização (Dados Originais):** Usa os números decimais puros, demonstrando visualmente como o algoritmo ID3 sofre de *overfitting* sem a devida preparação.
- **Teste B: Discretização por Largura Igual:** Divide a régua de valores em 2 faixas de tamanho exatos.
- **Teste C: Discretização por Frequência (Quantis):** Equilibra os dados para que cada grupo tenha a mesma quantidade de amostras.

In [ ]:
def calcular_profundidade(node):
    if not node.children:
        return 0
    return 1 + max([calcular_profundidade(filho) for filho in node.children.values()])

# 1. Carregar os dados iniciais
caminho_iris = "datasets/iris.csv" 
df_iris = pd.read_csv(caminho_iris)

# --- Traduzir colunas para Português ---
dicionario_traducao = {
    'sepallength': 'Comp. Sépala',
    'sepalwidth': 'Largura Sépala',
    'petallength': 'Comp. Pétala',
    'petalwidth': 'Largura Pétala',
    'class': 'Classe'
}
df_iris = df_iris.rename(columns=dicionario_traducao)

# Atualizar as variáveis com os novos nomes em português
colunas_features = ['Comp. Sépala', 'Largura Sépala', 'Comp. Pétala', 'Largura Pétala']
nome_da_coluna_alvo = 'Classe'
df_iris = df_iris[colunas_features + [nome_da_coluna_alvo]]


# ----------------- TESTE 1: CAOS (Dados Contínuos) -----------------
arvore_continua = DecisionTreeID3(max_depth=5)
arvore_continua.fit(df_iris, target_name=nome_da_coluna_alvo)
metricas_teste_1 = calcular_metricas(arvore_continua, df_iris, nome_da_coluna_alvo)

# --> Calcula e adiciona a profundidade!
metricas_teste_1['profundidade'] = calcular_profundidade(arvore_continua.root) 

plotar_arvore_decisao(arvore_continua.root, 
                      titulo="Árvore 1: Sem Discretização (Cenário de Overfitting Extremo)", 
                      metricas=metricas_teste_1)

# ----------------- TESTE 2: LARGURA IGUAL -----------------
df_ingenua = discretizar_largura_igual(df_iris, colunas_features)
arvore_ingenua = DecisionTreeID3(max_depth=5)
arvore_ingenua.fit(df_ingenua, target_name=nome_da_coluna_alvo)
metricas_teste_2 = calcular_metricas(arvore_ingenua, df_ingenua, nome_da_coluna_alvo)

# --> Calcula e adiciona a profundidade!
metricas_teste_2['profundidade'] = calcular_profundidade(arvore_ingenua.root)

plotar_arvore_decisao(arvore_ingenua.root, 
                      titulo="Árvore 2: Discretização Ingénua (Largura Igual)", 
                      metricas=metricas_teste_2)

# ----------------- TESTE 3: QUANTIS (Frequência Igual) -----------------
df_inteligente = discretizar_frequencia_igual(df_iris, colunas_features)
arvore_inteligente = DecisionTreeID3(max_depth=5)
arvore_inteligente.fit(df_inteligente, target_name=nome_da_coluna_alvo)
metricas_teste_3 = calcular_metricas(arvore_inteligente, df_inteligente, nome_da_coluna_alvo)

# --> Calcula e adiciona a profundidade!
metricas_teste_3['profundidade'] = calcular_profundidade(arvore_inteligente.root)

plotar_arvore_decisao(arvore_inteligente.root, 
                      titulo="Árvore 3: Discretização Inteligente (Frequência Igual)", 
                      metricas=metricas_teste_3)

### **Conclusão**
A nossa experiência demonstra claramente a importância da preparação dos dados. A Árvore 1 (sem discretização) obteve 100% de acurácia, mas é um caso clássico de overfitting: a árvore memorizou o treino e não conseguiria prever dados novos. A Árvore 2 (Largura Igual) evitou o overfitting, mas a sua divisão cega da "régua" de valores misturou espécies diferentes, afundando a acurácia para 80%.

A Árvore 3 (Frequência Igual / Quantis) foi a abordagem ideal. Ao respeitar a densidade e a distribuição real dos dados, o algoritmo ID3 conseguiu maximizar o Ganho de Informação, gerando um modelo compacto, sem overfitting e com uma excelente acurácia de 98%.

---

#### *Gerador da árvore criada pelo Monte Carlos*

In [ ]:
# 1. Função matemática para calcular a profundidade real do modelo
def calcular_profundidade(node):
    if not node.children:
        return 0
    return 1 + max([calcular_profundidade(filho) for filho in node.children.values()])

# 2. O "Remendo" para evitar o erro do 'branch_counts'
def corrigir_arvore_antiga(node):
    if not hasattr(node, 'branch_counts'):
        node.branch_counts = {}
    for filho in node.children.values():
        corrigir_arvore_antiga(filho)

# 3. A SUPER FUNÇÃO: Carrega, Testa e Plota!
def avaliar_e_plotar_arvore(nome_arvore, caminho_csv_teste):
    caminho_pkl = f"modelos_treinados/{nome_arvore}.pkl"
    
    try:
        # --- A. CARREGAR A ÁRVORE ---
        with open(caminho_pkl, 'rb') as f:
            arvore = pickle.load(f)
            
        corrigir_arvore_antiga(arvore.root)
        prof_maxima = calcular_profundidade(arvore.root)
        print(f"[OK] Árvore '{nome_arvore}' carregada (Profundidade: {prof_maxima})")

        # --- B. PREPARAR OS DADOS DE TESTE ---
        print(f"A carregar dados de teste: {caminho_csv_teste}...")
        df_teste = pd.read_csv(caminho_csv_teste)
        df_teste = df_teste[(df_teste['winner'] != 0) & (df_teste['p_turn'] == df_teste['winner'])]
        
        colunas_tabuleiro = [f"pos_{i}" for i in range(42)]
        for col in colunas_tabuleiro:
            df_teste[col] = np.where(df_teste[col] == 0, 0, 
                      np.where(df_teste[col] == df_teste['p_turn'], 1, -1))
            
        df_teste["target_move"] = df_teste["col"].astype(str) + "_" + df_teste["type"]
        df_teste_clean = df_teste.drop(columns=["game_id", "winner", "col", "type", "p_turn"])
        df_teste_clean = clean_conflicting_data(df_teste_clean, target_name="target_move")

        # --- C. CALCULAR MÉTRICAS (Acurácia) ---
        print("A calcular a inteligência da IA...")
        minhas_metricas = calcular_metricas(arvore, df_teste_clean, "target_move")
        
        # Juntar a profundidade às métricas de acurácia para aparecer tudo no gráfico!
        minhas_metricas['profundidade'] = prof_maxima

        # --- D. GERAR O GRÁFICO ---
        print(f"A gerar o gráfico final... (Pode demorar alguns segundos)")
        plotar_arvore_decisao(arvore.root, 
                              titulo=f"Visão e Avaliação da Árvore: {nome_arvore}", 
                              metricas=minhas_metricas)
        
    except FileNotFoundError:
        print(f"[ERRO] Não encontrei o ficheiro. Verifica o nome!")

# =======================================================
# EXECUÇÃO: Põe o nome da árvore e o ficheiro de teste
# =======================================================

# ATENÇÃO: Substitui o caminho do CSV abaixo por um ficheiro que o MCTS gerou 
# e que a árvore nunca tenha visto!
ARVORE_PARA_TESTAR = "best_tree_all"
DATASET_DE_TESTE = "datasets\P1_it10000_c1.41_mcAll_dNone_vs_P2_it10000_c1.41_mcAll_dNone.csv"  

avaliar_e_plotar_arvore(ARVORE_PARA_TESTAR, DATASET_DE_TESTE)

## **6. Arena de Benchmark: Torneio Automático de IA**

Nesta secção, implementamos um motor de torneio para colocar os diferentes modelos de IA a competir diretamente entre si (por exemplo, a Árvore de Decisão treinada contra o MCTS original). 

O objetivo aqui não é visualizar o tabuleiro turno a turno, mas sim executar dezenas ou centenas de partidas de forma rápida e silenciosa. No final da simulação, os resultados de cada jogo (quem venceu, se houve empate e o número total de jogadas) são automaticamente guardados num *dataset* (ficheiro CSV). Isto permite-nos extrair métricas sólidas para comparar o desempenho, a eficiência e a taxa de vitórias de cada abordagem no nosso relatório final.

In [ ]:
PASTA_MODELOS = "modelos_treinados"
PASTA_SAIDA   = "resultados"

# --- Árvores: alias → nome do ficheiro .pkl (sem extensão) ---
TREES = {
    "tree1": "best_tree_all",
    "tree2": "best_tree_win",
    "tree3": "worst_tree_all",
    "tree4": "worst_tree_win"
}

# --- MCTS: alias → parâmetros ---
MCTS_CONFIGS = {
    "mcts_best": dict(iterations=10000, c=1.41, max_children=None, max_depth=None, pure_mode=False),
    "mcts_standard": dict(iterations=10000, c=1.41, max_children=None, max_depth=None, pure_mode=True),
    "mcts_maxchildren_4": dict(iterations=10000, c=1.41, max_children=4, max_depth=None, pure_mode=True),
    "mcts_maxdepth_15": dict(iterations=10000, c=1.41, max_children=None, max_depth=15, pure_mode=True),
    "mcts_highC_3": dict(iterations=10000, c=3, max_children=None, max_depth=None, pure_mode=True),
    "mcts_lowC_1": dict(iterations=10000, c=1, max_children=None, max_depth=None, pure_mode=True),
}

NUM_JOGOS_POR_DUELO = 50

# ==============================================================
# 1. CARREGAMENTO E NOMES DESCRITIVOS
# ==============================================================
def _carregar_todos_os_agentes():
    agentes = {}
    nomes   = {}

    for alias, nome_ficheiro in TREES.items():
        caminho = os.path.join(PASTA_MODELOS, f"{nome_ficheiro}.pkl")
        try:
            with open(caminho, "rb") as f:
                agentes[alias] = pickle.load(f)
            nomes[alias] = nome_ficheiro
            print(f"[OK] Árvore carregada: {nome_ficheiro}")
        except FileNotFoundError:
            print(f"[AVISO] Árvore não encontrada (será ignorada no torneio): {caminho}")
            agentes[alias] = None

    for alias, params in MCTS_CONFIGS.items():
        agentes[alias] = MCTS(**params)
        nomes[alias]   = alias
        print(f"[OK] MCTS carregado: {alias}")

    return agentes, nomes

# ==============================================================
# 2. MOTOR DO DUELO (Grava no Ficheiro Global)
# ==============================================================
def run_matchup(num_games, ia_1, ia_2, name_ia1, name_ia2, output_file):
    print(f"\n[A COMBATER] {name_ia1} VS {name_ia2} (A jogar {num_games} partidas)...")
    vitorias_p1, vitorias_p2, empates = 0, 0, 0
    start_time = time.time()

    for i in range(num_games):
        game = PopOutGame()
        num_moves = 0

        ia_1_comeca = (i % 2 == 0)
        if ia_1_comeca:
            agente_player1, agente_player2 = ia_1, ia_2
            nome_quem_comeca = name_ia1
        else:
            agente_player1, agente_player2 = ia_2, ia_1
            nome_quem_comeca = name_ia2

        while True:
            # LIMITE DE 100 TURNOS E REPETIÇÃO PARA POUPAR TEMPO
            if game.check_repetition() or num_moves > 100:
                winner_player = 0
                break

            curr_ia = agente_player1 if game.current_player == PLAYER1 else agente_player2
            move = _get_ai_move(curr_ia, game)

            if move is None:
                winner_player = 0
                break

            col, m_type = move
            if m_type == 'd':
                game.drop_piece(col, game.current_player)
            else:
                game.pop_piece(col, game.current_player)

            num_moves += 1
            vencedor_atual = game.check_winner_after_move(game.current_player)
            if vencedor_atual is not None:
                winner_player = vencedor_atual
                break

            game.current_player = PLAYER2 if game.current_player == PLAYER1 else PLAYER1

        if winner_player == 0:
            empates += 1
            resultado_str = "Empate"
        elif (winner_player == PLAYER1 and ia_1_comeca) or (winner_player == PLAYER2 and not ia_1_comeca):
            vitorias_p1 += 1
            resultado_str = name_ia1
        else:
            vitorias_p2 += 1
            resultado_str = name_ia2

        df_partida = pd.DataFrame([{
            "P1": name_ia1, 
            "P2": name_ia2,
            "Vencedor": resultado_str,
            "Quem_Comecou": nome_quem_comeca,
            "Total_Jogadas": num_moves
        }])
        
        file_exists = os.path.isfile(output_file)
        # O mode="a" garante que não apaga o torneio_2.csv existente, apenas adiciona no fim!
        df_partida.to_csv(output_file, mode="a", header=not file_exists, index=False)

    elapsed = time.time() - start_time
    print(f" -> Resultado desta sessão: {name_ia1} [{vitorias_p1}] | {name_ia2} [{vitorias_p2}] | Empates [{empates}] (Demorou {elapsed:.1f}s)")

# ==============================================================
# 3. MODO TORNEIO COMPLETO (ROUND-ROBIN)
# ==============================================================
print("A carregar e validar agentes...\n")
CATALOGO, NOMES = _carregar_todos_os_agentes()

# Remove as Árvores que eventualmente não tenham o ficheiro gerado
agentes_validos = {alias: agente for alias, agente in CATALOGO.items() if agente is not None}

print(f"\n" + "="*50)
print(f" LIGA DOS CAMPEÕES POPOUT AI - TORNEIO 2")
print(f"="*50)

os.makedirs(PASTA_SAIDA, exist_ok=True)
FICHEIRO_TORNEIO = os.path.join(PASTA_SAIDA, "torneio_2.csv")

# Gera todas as combinações de duelos 1v1
import itertools
todos_os_duelos = list(itertools.combinations(agentes_validos.keys(), 2))

tempo_total_inicio = time.time()

for alias_1, alias_2 in todos_os_duelos:
    ia_1 = agentes_validos[alias_1]
    ia_2 = agentes_validos[alias_2]
    nome_1 = NOMES[alias_1]
    nome_2 = NOMES[alias_2]
    
    run_matchup(
        num_games=NUM_JOGOS_POR_DUELO, 
        ia_1=ia_1, ia_2=ia_2, 
        name_ia1=nome_1, name_ia2=nome_2, 
        output_file=FICHEIRO_TORNEIO
    )

print("\n" + "="*60)
print(f"TORNEIO 2 CONCLUÍDO! (Tempo Total: {(time.time() - tempo_total_inicio)/60:.2f} minutos)")
print(f"O ficheiro foi gravado com sucesso em: {FICHEIRO_TORNEIO}")
print("="*60)


#### **Gráfico**

In [ ]:
# 1. Carregar os resultados
df = pd.read_csv(r"E:resultados\P1_it1000_c1.41_mcAll_d10_vs_P2_it1000_c1.41_mcAll_dNone.csv")

# 2. Configurar o estilo visual
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5)) # Tamanho ajustado para apenas um gráfico

# --- Gráfico: Taxa de Vitórias Global ---
# Atribuímos o gráfico à variável 'ax' para poder manipulá-lo
ax = sns.countplot(data=df, x='Vencedor', palette="viridis")

plt.title("Taxa de Vitórias Global", fontweight='bold')
plt.xlabel("Resultado / Vencedor")
plt.ylabel("Número de Partidas")

# --- Adicionar os totais em cima das barras ---
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()